# 🏋️ Training: Bangla Image Captioning (SigLIP2 + GRU + LSTM + BanglaGPT)

This notebook trains a multi-decoder image captioning model for Bengali (Bangla).

**Architecture Overview:**
```
Image → SigLIP2 (Vision Encoder) → Linear Projection → GRU → LSTM → BanglaGPT → Caption
```

**Pipeline:**
1. Load consolidated dataset40k (images + captions)
2. Build vocabulary from training captions
3. Create DataLoaders with augmentation
4. Train with cross-entropy loss + label smoothing
5. Validate and save checkpoints


## 📥 Imports & Environment Setup


In [1]:
import os, sys, json, time, random, logging, warnings
from datetime import datetime
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from bangla_dataset import BanglaCaptionDataset, Vocabulary
from torchvision import transforms

from transformers import AutoModel, AutoImageProcessor
from transformers import get_cosine_schedule_with_warmup

from PIL import Image
from tqdm import tqdm
from dataclasses import dataclass
import pandas as pd


C:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0615 19:33:54.169000 17636 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0615 19:33:54.254000 17636 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
# Suppress non-critical warnings for clean output
warnings.filterwarnings("ignore")
os.environ["TORCH_CPP_LOG_LEVEL"] = "ERROR"
os.environ["TORCH_DISTRIBUTED_DEBUG"] = "OFF"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)
logging.getLogger("torch._inductor").setLevel(logging.ERROR)


## ⚙️ Configuration

All hyperparameters and paths are defined here. Adjust as needed.


In [3]:
# 📂 Paths - auto-detect experiment root
import os as _os
_cwd = _os.getcwd()
for _ in range(4):
    if _os.path.isdir(_os.path.join(_cwd, "output")):
        break
    # Also check subdirectories (handles running from project root)
    _found_exp = False
    try:
        for _entry in _os.listdir(_cwd):
            _sub = _os.path.join(_cwd, _entry)
            if _os.path.isdir(_sub) and _os.path.isdir(_os.path.join(_sub, "output")):
                _cwd = _sub
                _found_exp = True
                break
    except PermissionError:
        pass
    if _found_exp:
        break
    _p = _os.path.dirname(_cwd)
    if _p == _cwd:
        break
    _cwd = _p
    if _os.path.isdir(_os.path.join(_cwd, "output")):
        break
    _p = _os.path.dirname(_cwd)
    if _p == _cwd:
        break
    _cwd = _p
MODEL_FOLDER = _cwd
os.environ["HF_HOME"] = os.path.join(MODEL_FOLDER, "hf_cache")

DATA_DIR = os.path.join(MODEL_FOLDER, "dataset40k")
MODELS_DIR = os.path.join(MODEL_FOLDER, "models")
OUTPUT_DIR = os.path.join(MODEL_FOLDER, "output")
SIGLIP_MODEL_PATH = os.path.join(MODELS_DIR, "siglip2-base-patch32-256")
BANGLA_GPT_PATH = os.path.join(MODELS_DIR, "BanglaGPT")


In [4]:
# 🖥️ Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# 📐 Model dimensions
FEATURE_DIM = 768      # SigLIP2 output dimension
EMBED_DIM = 512        # Word embedding dimension
HIDDEN_DIM = 512       # RNN hidden dimension
DROPOUT = 0.3333
BANGLA_GPT_HIDDEN = 768

# 🔢 Architecture layers
GRU_NUM_LAYERS = 4
LSTM_NUM_LAYERS = 4
UNFREEZE_VISION_LAYERS = 4   # Fine-tune last N vision layers
UNFREEZE_GPT_LAYERS = 4      # Fine-tune last N GPT layers


In [5]:
# 📊 Training hyperparameters
BATCH_SIZE = 64
ACCUMULATION_STEPS = 1
EPOCHS = 20
MAX_LR = 5e-5
MAX_SEQ_LEN = 26       # Overwritten at runtime with p99
USE_AMP = True          # Automatic Mixed Precision
NUM_WORKERS = 6
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 0.01

# 🔀 Data split ratios
TRAIN_RATIO = 0.9
VAL_RATIO = 0.05
TEST_RATIO = 0.05
TRAIN_SEED = 1029384756

BANGLA_GPT_NAME = "shahidul034/BanglaGPT"


## 📝 Model Output Data Class


In [6]:
@dataclass
class ModelOutput:
    """Typed container for model forward output."""
    loss: torch.Tensor
    logits: torch.Tensor


## 🔤 Vocabulary

Manages token↔word mapping for Bengali text. Builds from training captions with configurable minimum frequency.


## 🧠 Model Architecture

### Attention Layer
Bahdanau-style additive attention mechanism for focusing on relevant image regions.


In [7]:
class AttentionLayer(nn.Module):
    """Bahdanau attention mechanism."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.attn_W = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, decoder_output, encoder_outputs):
        adapted = self.attn_W(encoder_outputs)
        scores = torch.bmm(decoder_output, adapted.transpose(1, 2))
        alignment = torch.softmax(scores, dim=-1)
        return torch.bmm(alignment, encoder_outputs)


### GRU Decoder
First stage: processes embedded tokens with a multi-layer GRU + attention.


In [8]:
class GRUDecoder(nn.Module):
    """GRU-based decoder with attention — first stage of decoding pipeline."""

    def __init__(self, vocab_size, embedding_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.GRU(
            embedding_dim, hidden_dim, num_layers=GRU_NUM_LAYERS, batch_first=True
        )
        self.attention = AttentionLayer(hidden_dim)
        self.concat = nn.Linear(hidden_dim * 2, hidden_dim)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, encoder_outputs, input_ids, hidden=None):
        batch_size = input_ids.size(0)
        device = input_ids.device
        embeddings = self.dropout(self.embedding(input_ids))
        if hidden is None:
            hidden = torch.zeros(GRU_NUM_LAYERS, batch_size, HIDDEN_DIM, device=device)
        rnn_outputs, hidden = self.rnn(embeddings, hidden)
        context = self.attention(rnn_outputs, encoder_outputs)
        combined = torch.cat([context, rnn_outputs], dim=-1)
        fused = self.tanh(self.concat(combined))
        return self.dropout(fused)


### LSTM Decoder
Second stage: refines GRU outputs through an LSTM with attention.


In [9]:
class LSTMDecoder(nn.Module):
    """LSTM-based decoder with attention — second stage of decoding pipeline."""

    def __init__(self, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.rnn = nn.LSTM(
            hidden_dim, hidden_dim, num_layers=LSTM_NUM_LAYERS, batch_first=True
        )
        self.attention = AttentionLayer(hidden_dim)
        self.concat = nn.Linear(hidden_dim * 2, hidden_dim)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, encoder_outputs, gru_features, hidden=None):
        batch_size = gru_features.size(0)
        device = gru_features.device
        if hidden is None:
            h0 = torch.zeros(LSTM_NUM_LAYERS, batch_size, HIDDEN_DIM, device=device)
            c0 = torch.zeros(LSTM_NUM_LAYERS, batch_size, HIDDEN_DIM, device=device)
            hidden = (h0, c0)
        rnn_outputs, hidden = self.rnn(gru_features, hidden)
        context = self.attention(rnn_outputs, encoder_outputs)
        combined = torch.cat([context, rnn_outputs], dim=-1)
        fused = self.tanh(self.concat(combined))
        return self.dropout(fused)


### BanglaGPT Decoder
Third stage: feeds LSTM features + embeddings into a frozen (partially unfrozen) BanglaGPT model for fluent caption generation.


In [10]:
class BanglaGPTDecoder(nn.Module):
    """BanglaGPT-based decoder — third stage, generates final token logits."""

    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.input_proj = nn.Linear(HIDDEN_DIM + EMBED_DIM, BANGLA_GPT_HIDDEN)
        model_path = (
            BANGLA_GPT_PATH if os.path.exists(BANGLA_GPT_PATH) else BANGLA_GPT_NAME
        )
        self.gpt = AutoModel.from_pretrained(model_path, trust_remote_code=True).float()
        for param in self.gpt.parameters():
            param.requires_grad = False

        # Optionally unfreeze the last N GPT layers
        if UNFREEZE_GPT_LAYERS > 0:
            gpt_layers = getattr(self.gpt, "h", None)
            if gpt_layers is None:
                gpt_layers = getattr(getattr(self.gpt, "transformer", None), "h", None)
            if gpt_layers is not None and hasattr(gpt_layers, "__len__"):
                total = len(gpt_layers)
                start = max(0, total - UNFREEZE_GPT_LAYERS)
                for i in range(start, total):
                    for p in gpt_layers[i].parameters():
                        p.requires_grad = True
                print(f"BanglaGPT: Unfreezing last {UNFREEZE_GPT_LAYERS}/{total} layers")

        self.output_proj = nn.Linear(BANGLA_GPT_HIDDEN, vocab_size)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, lstm_features, input_ids):
        text_embeds = self.embedding(input_ids)
        combined = torch.cat([lstm_features, text_embeds], dim=-1)
        gpt_inputs = self.input_proj(combined)
        gpt_out = self.gpt(inputs_embeds=gpt_inputs).last_hidden_state
        return self.output_proj(self.dropout(gpt_out))


### Multi-Decoder Fusion
Chains GRU → LSTM → BanglaGPT into a single sequential pipeline.


In [11]:
class MultiDecoderFusion(nn.Module):
    """Chains GRU → LSTM → BanglaGPT decoders sequentially."""

    def __init__(self, vocab_size):
        super().__init__()
        self.gru = GRUDecoder(vocab_size)
        self.lstm = LSTMDecoder()
        self.banglagpt = BanglaGPTDecoder(vocab_size)

    def forward(self, encoder_outputs, input_ids):
        gru_features = self.gru(encoder_outputs, input_ids)
        lstm_features = self.lstm(encoder_outputs, gru_features)
        return self.banglagpt(lstm_features, input_ids)


### SigLIP2 Vision Encoder
Vision encoder using SigLIP2, with optional unfreezing of the last N layers.


In [12]:
class Encoder(nn.Module):
    """SigLIP2 vision encoder with optional fine-tuning of last layers."""

    def __init__(self, unfreeze_layers=UNFREEZE_VISION_LAYERS):
        super().__init__()
        model_path = (
            SIGLIP_MODEL_PATH if os.path.exists(SIGLIP_MODEL_PATH)
            else "google/siglip2-base-patch32-256"
        )
        self.siglip = AutoModel.from_pretrained(model_path, trust_remote_code=True)
        self.unfreeze_layers = unfreeze_layers
        self._fully_frozen = True
        self._configure_gradients()

    def _configure_gradients(self):
        """Configure which vision layers are trainable."""
        if self.unfreeze_layers == 0:
            for param in self.siglip.parameters():
                param.requires_grad = True
            self._fully_frozen = False
        elif self.unfreeze_layers > 0:
            for param in self.siglip.parameters():
                param.requires_grad = False
            layers = self.siglip.vision_model.encoder.layers
            start = max(0, len(layers) - self.unfreeze_layers)
            for i in range(start, len(layers)):
                for param in layers[i].parameters():
                    param.requires_grad = True
            self._fully_frozen = False
        else:
            # Fully frozen
            for param in self.siglip.parameters():
                param.requires_grad = False

    def forward(self, pixel_values):
        if self._fully_frozen:
            with torch.no_grad():
                return self.siglip.vision_model(pixel_values).last_hidden_state
        return self.siglip.vision_model(pixel_values).last_hidden_state


### Full CaptionModel
Complete model: Encoder → Vision Projection → MultiDecoderFusion with loss computation and caption generation.


In [13]:
class CaptionModel(nn.Module):
    """Complete image captioning model: Encoder → Projection → MultiDecoder."""

    def __init__(self, vocab_size, unfreeze_vision=UNFREEZE_VISION_LAYERS):
        super().__init__()
        self.encoder = Encoder(unfreeze_layers=unfreeze_vision)
        self.vision_projection = nn.Linear(FEATURE_DIM, HIDDEN_DIM)
        self.decoder = MultiDecoderFusion(vocab_size)
        self.loss_fn = nn.CrossEntropyLoss(
            ignore_index=-100, label_smoothing=LABEL_SMOOTHING
        )

    def forward(self, pixel_values, input_ids, labels=None):
        features = self.encoder(pixel_values)
        visual_tokens = self.vision_projection(features)
        logits = self.decoder(visual_tokens, input_ids)
        if labels is None:
            labels = input_ids.clone()
            labels[labels == 0] = -100
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()
        loss = self.loss_fn(
            shift_logits.view(-1, logits.size(-1)), shift_labels.view(-1)
        )
        return ModelOutput(loss=loss, logits=logits)

    @torch.no_grad()
    def generate_captions(self, pixel_values, vocab, max_new_tokens=MAX_SEQ_LEN):
        """Greedy decoding for batched caption generation."""
        self.eval()
        if pixel_values.dim() == 3:
            pixel_values = pixel_values.unsqueeze(0)
        features = self.encoder(pixel_values)
        visual_tokens = self.vision_projection(features)
        batch_size = pixel_values.size(0)
        device = pixel_values.device
        start_id, end_id = vocab.word2idx["<start>"], vocab.word2idx["<end>"]
        generated = torch.full(
            (batch_size, 1), start_id, dtype=torch.long, device=device
        )

        unk_id = vocab.word2idx.get("<unk>", 3)
        for _ in range(max_new_tokens - 1):
            logits = self.decoder(visual_tokens, generated)
            logits[:, -1, unk_id] = -1e9
            next_token = logits[:, -1:, :].argmax(dim=-1)
            generated = torch.cat([generated, next_token], dim=1)
            if (next_token == end_id).all():
                break
        return [vocab.decode(seq.tolist()) for seq in generated]

    def generate_caption(self, pixel_values, vocab, max_new_tokens=MAX_SEQ_LEN):
        """Generate a single caption (wrapper around generate_captions)."""
        return self.generate_captions(pixel_values, vocab, max_new_tokens)[0]

    @torch.no_grad()
    def generate_beam(self, pixel_values, vocab, beam_size=5, max_new_tokens=MAX_SEQ_LEN):
        """Beam search decoding for improved caption quality."""
        self.eval()
        if pixel_values.dim() == 3:
            pixel_values = pixel_values.unsqueeze(0)
        features = self.encoder(pixel_values)
        visual_tokens = self.vision_projection(features)
        start_id = vocab.word2idx["<start>"]
        end_id = vocab.word2idx["<end>"]

        sequences = [[start_id]]
        scores = [0.0]

        for _ in range(max_new_tokens):
            all_candidates = []
            for i, seq in enumerate(sequences):
                if seq[-1] == end_id:
                    all_candidates.append((scores[i], seq))
                    continue
                ids = torch.tensor([seq], device=pixel_values.device)
                logits = self.decoder(visual_tokens, ids)[:, -1, :]
                log_probs = torch.log_softmax(logits, dim=-1)[0]
                top = log_probs.topk(beam_size)
                for lp, tok in zip(top.values, top.indices):
                    all_candidates.append((scores[i] + lp.item(), seq + [tok.item()]))

            ordered = sorted(all_candidates, key=lambda x: x[0], reverse=True)
            sequences = [c[1] for c in ordered[:beam_size]]
            scores = [c[0] for c in ordered[:beam_size]]

            if all(s[-1] == end_id for s in sequences):
                break

        return vocab.decode(sequences[0])


## 📂 Data Loading

### Load Captions
Parse caption.txt into a dictionary mapping filenames → list of captions.


In [14]:
def load_captions(caption_file):
    """Load captions from file (format: filename caption)."""
    captions = {}
    with open(caption_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                fname, cap = parts[0], parts[1]
                if fname not in captions:
                    captions[fname] = []
                captions[fname].append(cap)
    return captions


### Dataset Class
PyTorch Dataset with optional augmentation (random horizontal flip + color jitter) for training.


### Training Log Saver
Saves training progress to an Excel (or CSV) file.


In [15]:
def save_training_log(log_file, epoch, train_loss, val_loss, lr, time_taken):
    """Append one row of training metrics to a log file."""
    df_new = pd.DataFrame({
        "Epoch": [epoch],
        "Train Loss": [train_loss],
        "Val Loss": [val_loss],
        "Learning Rate": [f"{lr:.10f}"],
        "Time (s)": [time_taken],
    })
    fallback = log_file.replace(".xlsx", ".csv")
    try:
        if os.path.exists(log_file):
            pd.concat([pd.read_excel(log_file), df_new], ignore_index=True).to_excel(
                log_file, index=False
            )
        else:
            df_new.to_excel(log_file, index=False)
    except Exception:
        if os.path.exists(fallback):
            pd.concat([pd.read_csv(fallback), df_new], ignore_index=True).to_csv(
                fallback, index=False
            )
        else:
            df_new.to_csv(fallback, index=False)


## 🚀 Training Loop

The main `train()` function orchestrates:
- Data splitting (train/val/test)
- Vocabulary building with length distribution analysis
- Model initialization with layer-specific learning rates
- Training loop with gradient accumulation, AMP, and cosine scheduling
- Validation after each epoch with sample caption generation
- Checkpoint saving with best-model tracking


In [16]:
def train():
    """Main training function."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    log_file = os.path.join(
        OUTPUT_DIR, f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    )

    print("Loading data...")
    captions = load_captions(os.path.join(DATA_DIR, "caption.txt"))
    image_names = list(captions.keys())
    n = len(image_names)

    # Data splitting with fixed seed
    generator = torch.Generator().manual_seed(TRAIN_SEED)
    perm = torch.randperm(n, generator=generator).tolist()
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)
    train_list = [image_names[i] for i in perm[:n_train]]
    val_list = [image_names[i] for i in perm[n_train:n_train + n_val]]
    test_list = [image_names[i] for i in perm[n_train + n_val:]]

    print(f"\n{'=' * 50}")
    for name, lst in [("Train", train_list), ("Val", val_list), ("Test", test_list)]:
        print(f"{name:<12} {len(lst):<10} {len(lst) / n * 100:.2f}%")
    print(f"{'=' * 50}\nImages: {len(captions)}")

    # Analyze caption lengths to set MAX_SEQ_LEN
    all_lengths = [len(cap.split()) + 2 for caps in captions.values() for cap in caps]
    all_lengths.sort()
    nl = len(all_lengths)
    p99 = all_lengths[int(nl * 0.99)]
    global MAX_SEQ_LEN
    MAX_SEQ_LEN = p99
    print(f"Setting MAX_SEQ_LEN = {MAX_SEQ_LEN}")

    print("Building word-level vocabulary from training captions...")
    train_captions = {k: captions[k] for k in train_list}
    vocab = Vocabulary()
    vocab.build_from_captions(train_captions, min_freq=1)
    print(f"Vocabulary size: {vocab.vocab_size}")
    vocab_path = os.path.join(OUTPUT_DIR, "vocab.json")
    vocab.save(vocab_path)

    print("Creating datasets...")
    siglip_path = (
        SIGLIP_MODEL_PATH if os.path.exists(SIGLIP_MODEL_PATH)
        else "google/siglip2-base-patch32-256"
    )
    train_dataset = BanglaCaptionDataset(
        train_list, captions, vocab, os.path.join(DATA_DIR, "images"), siglip_path, is_train=True, max_len=MAX_SEQ_LEN
    )
    val_dataset = BanglaCaptionDataset(
        val_list, captions, vocab, os.path.join(DATA_DIR, "images"), siglip_path, is_train=False, max_len=MAX_SEQ_LEN
    )

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )

    # Initialize model
    model = CaptionModel(vocab_size=vocab.vocab_size).to(DEVICE)
    if torch.__version__.startswith("2"):
        model = torch.compile(model)

    # Separate parameter groups with different learning rates
    vision_params = [p for p in model.encoder.parameters() if p.requires_grad]
    proj_params = list(model.vision_projection.parameters())
    decoder_params = list(model.decoder.parameters())
    all_params = vision_params + proj_params + decoder_params
    total_trainable = sum(p.numel() for p in all_params)

    optimizer = torch.optim.AdamW([
        {"params": vision_params, "lr": 5e-5},
        {"params": proj_params, "lr": 2e-4},
        {"params": decoder_params, "lr": 3e-4},
    ], weight_decay=WEIGHT_DECAY)

    scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

    total_steps = (
        (len(train_loader) + ACCUMULATION_STEPS - 1) // ACCUMULATION_STEPS * EPOCHS
    )
    warmup_steps = total_steps // 10
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    print(f"\n{'=' * 50}")
    print(f"Architecture: SigLIP2 → Linear(768,{HIDDEN_DIM}) → Sequential Pipeline")
    print(f"  ├─ GRU ({GRU_NUM_LAYERS}-layer + attention)")
    print(f"  ├─ LSTM ({LSTM_NUM_LAYERS}-layer + attention)")
    print(f"  └─ BanglaGPT → generates caption from LSTM features")
    print(f"Trainable: {total_trainable:,} | Vocab: {vocab.vocab_size:,} | MaxLen: {MAX_SEQ_LEN}")
    print(f"Device: {DEVICE} | Batch: {BATCH_SIZE * ACCUMULATION_STEPS}")
    print(f"{'=' * 50}\n")

    best_val_loss = float("inf")

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        model.train()
        total_loss = 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS}")
        running_loss = 0.0
        running_count = 0

        for batch_idx, (pixel_values, caption_ids) in enumerate(pbar):
            pixel_values = pixel_values.to(DEVICE)
            caption_ids = caption_ids.to(DEVICE)
            labels = caption_ids.clone()
            labels[labels == 0] = -100

            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                outputs = model(pixel_values, caption_ids, labels=labels)
                loss = outputs.loss / ACCUMULATION_STEPS

            if USE_AMP:
                scaler.scale(loss).backward()
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(all_params, max_norm=0.5)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()
            else:
                loss.backward()
                if (batch_idx + 1) % ACCUMULATION_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(all_params, max_norm=0.5)
                    optimizer.step()
                    optimizer.zero_grad()
                    scheduler.step()

            batch_loss = loss.item() * ACCUMULATION_STEPS
            total_loss += batch_loss
            running_loss += batch_loss
            running_count += 1
            pbar.set_postfix({"loss": f"{running_loss / running_count:.4f}"})

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for pixel_values, caption_ids in val_loader:
                pixel_values = pixel_values.to(DEVICE)
                caption_ids = caption_ids.to(DEVICE)
                labels = caption_ids.clone()
                labels[labels == 0] = -100
                with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                    outputs = model(pixel_values, caption_ids, labels=labels)
                val_loss += outputs.loss.float().item()

        avg_val_loss = val_loss / len(val_loader)
        epoch_time = time.time() - epoch_start
        current_lr = optimizer.param_groups[2]["lr"]

        # Sample caption generation for qualitative check
        sample_img, sample_cap = val_dataset[0]
        with torch.no_grad():
            sample_pv = sample_img.unsqueeze(0).to(DEVICE)
            pred = model.generate_caption(sample_pv, vocab)
        ref = vocab.decode(sample_cap.tolist())
        print(f"  REF : {ref}")
        print(f"  PRED: {pred}")

        print(f"Epoch {epoch + 1}/{EPOCHS} | {epoch_time:.1f}s | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f} | LR: {current_lr:.2e}\n")
        save_training_log(log_file, epoch + 1, avg_train_loss, avg_val_loss, current_lr, epoch_time)

        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_train_loss, "val_loss": avg_val_loss,
            }, os.path.join(OUTPUT_DIR, "best_model.pt"))
            print(f"  New best model saved (val_loss: {best_val_loss:.4f})")

        # Save epoch checkpoint (keep last 3)
        ckpt_path = os.path.join(OUTPUT_DIR, f"checkpoint_epoch_{epoch + 1}.pt")
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": avg_train_loss, "val_loss": avg_val_loss,
        }, ckpt_path)
        prev = epoch + 1 - 3
        if prev > 0:
            old = os.path.join(OUTPUT_DIR, f"checkpoint_epoch_{prev}.pt")
            if os.path.exists(old):
                os.remove(old)

    # Save final model
    torch.save({"model_state_dict": model.state_dict()},
           os.path.join(OUTPUT_DIR, "final_model.pt"))
    print("Training complete!")


## ▶️ Run Training

Execute the training loop with the configured hyperparameters.


In [17]:
if __name__ == "__main__":
    train()


Loading data...

Train        39740      90.00%
Val          2207       5.00%
Test         2209       5.00%
Images: 44156
Setting MAX_SEQ_LEN = 26
Building word-level vocabulary from training captions...
Vocabulary size: 34112
Creating datasets...


Loading weights: 100%|██████████████████████████████| 148/148 [00:00<00:00, 619.65it/s, Materializing param=wte.weight]


BanglaGPT: Unfreezing last 4/12 layers

Architecture: SigLIP2 → Linear(768,512) → Sequential Pipeline
  ├─ GRU (4-layer + attention)
  ├─ LSTM (4-layer + attention)
  └─ BanglaGPT → generates caption from LSTM features
Trainable: 231,223,617 | Vocab: 34,113 | MaxLen: 26
Device: cuda | Batch: 64



Epoch 1/20: 100%|█████████████████████████████████████████████████████| 3068/3068 [09:41<00:00,  5.28it/s, loss=5.9407]


  REF : যখন কিছু লোক শস্যাগারের দিকে তাকায় অন্যরা সেতুর উপর দিয়ে হাঁটছে এবং কেউ কেউ সমুদ্র সৈকতের জলে শীতল হওয়া উপভোগ করছে
  PRED: একটি ছোট মেয়ে একটি সৈকতে একটি বড় পাথরের উপর দাঁড়িয়ে আছে
Epoch 1/20 | 688.3s | Train: 5.9407 | Val: 4.7970 | LR: 1.50e-04

  New best model saved (val_loss: 4.7970)


...

  REF : যখন কিছু লোক শস্যাগারের দিকে তাকায় অন্যরা সেতুর উপর দিয়ে হাঁটছে এবং কেউ কেউ সমুদ্র সৈকতের জলে শীতল হওয়া উপভোগ করছে
  PRED: একটি শিশু একটি বাড়ির পাশে একটি অগভীর পুকুরে খেলছে
Epoch 9/20 | 559.1s | Train: 3.3868 | Val: 4.2313 | LR: 2.01e-04



Epoch 10/20: 100%|████████████████████████████████████████████████████| 3068/3068 [08:06<00:00,  6.30it/s, loss=3.2644]


  REF : যখন কিছু লোক শস্যাগারের দিকে তাকায় অন্যরা সেতুর উপর দিয়ে হাঁটছে এবং কেউ কেউ সমুদ্র সৈকতের জলে শীতল হওয়া উপভোগ করছে
  PRED: একটি শিশু একটি ছোট নদীর তীরে ছুটে চলেছে
Epoch 10/20 | 565.9s | Train: 3.2644 | Val: 4.2629 | LR: 1.76e-04



Epoch 11/20:  20%|██████████▌                                          | 608/3068 [02:29<05:33,  7.37it/s, loss=3.0782]Exception ignored while calling deallocator <function _MultiProcessingDataLoaderIter.__del__ at 0x00000246D1002FB0>:
Traceback (most recent call last):
  File "C:\Python314\Lib\site-packages\torch\utils\data\dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "C:\Python314\Lib\site-packages\torch\utils\data\dataloader.py", line 1654, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "C:\Python314\Lib\multiprocessing\process.py", line 156, in join
    res = self._popen.wait(timeout)
  File "C:\Python314\Lib\multiprocessing\popen_spawn_win32.py", line 114, in wait
    res = _winapi.WaitForSingleObject(int(self._handle), msecs)
KeyboardInterrupt: 
Epoch 11/20:  20%|██████████▌                                          | 608/3068 [02:32<10:16,  3.99it/s, loss=3.0782]


KeyboardInterrupt: 